# 04 — MC-dropout uncertainty

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

Notebook 3 asked whether the *explanations* can be trusted. This notebook asks the matching
question about the *predictions*: **does the model know when it doesn't know?** It runs the
trained ResNet18 with its dropout layer left on at test time, takes 30 stochastic forward passes
per image, and turns the spread of those passes into per-image uncertainty scores.

Then it checks whether those scores are any use:
- **calibration:** when the model says 90%, is it right about 90% of the time?
- **error detection:** are the wrong predictions the uncertain ones?
- **structure:** is any class, or any tumor size, systematically less certain?

The dangerous case for a clinical tool is **confident and wrong**: a high-confidence error that no
uncertainty threshold would flag. Notebook 5 then joins this with notebook 3's audit to ask whether
*confident* predictions also come with *faithful* explanations.

**Inputs to attach** (Add Input → Your Work → Notebooks):

| input | used for | on `sys.path`? |
|---|---|---|
| notebook 0 output | `manifest.csv` + `.mat` files (MD5-matched) | no |
| notebook 1 output | checkpoint, `test_predictions.csv`, `src/` | **yes, its `src/` only** (copied, see §2b) |

Notebooks 2 and 3 aren't needed here. Uncertainty doesn't depend on the explanations. The outputs
are still built to join with them: same row order as `test_predictions.csv`, keyed by `file_path`,
with the checkpoint and manifest MD5s recorded.

**Outputs** (`/kaggle/working/results/uncertainty/`): `uncertainty_results.csv` (one row per test
image), `uncertainty_summary.json`, `mc_samples.npz` (all 30 × N × 3 softmax outputs), and
figures in `results/figures/`. `src/` (with the new `src/uncertainty/`) and `tests/` are saved too.

## 1. GPU check

The notebook stops here if there's no GPU. On a GPU, 30 passes over ~460 test images take well
under a minute.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Session options -> Accelerator -> GPU, then restart the session.")
DEVICE = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import copy
import hashlib
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_curve
from torch import nn
from torch.utils.data import DataLoader

WORK = Path("/kaggle/working")
UNC_DIR = WORK / "results" / "uncertainty"
FIG_DIR = WORK / "results" / "figures"
for d in [UNC_DIR, FIG_DIR, WORK / "tests"]:
    d.mkdir(parents=True, exist_ok=True)

## 2. Locate inputs

There must be exactly one `results/checkpoints/best_model.pt` under `/kaggle/input`, at any depth.
None, or several, stops the notebook; it never retrains. The manifest is the one whose MD5 equals
the `manifest_md5` stored in the checkpoint, so the test split here is provably the one notebook 1
evaluated.

In [ ]:
def find_unique(pattern, parent_names, what, how_to_fix):
    hits = [h for h in sorted(Path("/kaggle/input").rglob(pattern))
            if [p.name for p in h.parents[:len(parent_names)]] == parent_names]
    if len(hits) == 0:
        print("mounted under /kaggle/input:")
        for p in sorted(Path("/kaggle/input").glob("*/*"))[:40]:
            print("   ", p)
        raise FileNotFoundError(f"{what} not found under /kaggle/input. {how_to_fix}")
    if len(hits) > 1:
        raise RuntimeError(f"several copies of {what} attached, keep only one: {hits}")
    return hits[0]


CKPT_PATH = find_unique("best_model.pt", ["checkpoints", "results"], "results/checkpoints/best_model.pt",
                        "Attach notebook 1's committed output. Not retraining here.")
NB1_DIR = CKPT_PATH.parents[2]
for required in ["src/models/classifier.py", "src/data/dataset.py", "src/data/transforms.py",
                 "results/test_predictions.csv", "results/test_metrics.json"]:
    if not (NB1_DIR / required).exists():
        raise FileNotFoundError(f"notebook 1 output at {NB1_DIR} is missing {required} -- re-commit notebook 1")
CKPT_MD5 = hashlib.md5(CKPT_PATH.read_bytes()).hexdigest()

ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
WANT_MD5 = ckpt_raw["manifest_md5"]
MANIFEST_PATH = DATA_ROOT = None
for m in sorted(Path("/kaggle/input").rglob("manifest.csv")):
    if hashlib.md5(m.read_bytes()).hexdigest() != WANT_MD5:
        continue
    first = pd.read_csv(m, nrows=1).file_path.iloc[0]
    for root in [m.parent / "data", m.parent.parent / "data"]:
        if (root / first).exists():
            MANIFEST_PATH, DATA_ROOT = m, root
            break
    if MANIFEST_PATH:
        break
if MANIFEST_PATH is None:
    raise FileNotFoundError(f"no attached manifest.csv with md5 {WANT_MD5} (+ its data/ folder). "
                            "Attach the notebook-0 version notebook 1 was trained on.")
print("checkpoint:", CKPT_PATH, "| md5", CKPT_MD5)
print("manifest  :", MANIFEST_PATH, "(md5 matches checkpoint)")

### 2b. Code: notebook 1's `src/`, copied, plus `src/uncertainty/`

The same pattern as notebooks 2 and 3. Notebook 1's `src/` is the only code tree used. It's
copied into `/kaggle/working/src` because this notebook adds `src/uncertainty/` and inputs are
read-only. All `/kaggle/input` paths are removed from `sys.path`, and the assertion confirms `src`
resolves to `/kaggle/working/src`.

In [ ]:
shutil.copytree(NB1_DIR / "src", WORK / "src", dirs_exist_ok=True)
(WORK / "src" / "uncertainty").mkdir(exist_ok=True)
(WORK / "src" / "uncertainty" / "__init__.py").touch()

sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

import src
assert Path(src.__file__).resolve().parent == (WORK / "src").resolve(), src.__file__
print("src package:", Path(src.__file__).parent)

## 3. MC-dropout: what it is, and what this version can and can't claim

**The idea** (Gal & Ghahramani, 2016). A network trained with dropout can be read as an
approximate Bayesian model, with each dropout mask a sample of "plausible weights". Leaving dropout
on at test time and running `T` passes gives `T` different predictions for the same image; their
spread approximates how uncertain the model is about its own weights for that input.

**Getting the mode right is the part that's easy to break.** "Dropout on at inference" must mean
*only* dropout is in training mode:

- `model.eval()` first, then switch **only** the dropout modules back to `.train()`.
- **Never `model.train()`.** That would also put BatchNorm into training mode. BN would then
  normalise each batch with *that batch's* statistics, so an image's prediction would depend on
  its batch-mates, and every forward pass would overwrite the running mean/variance learned during
  training. The model would be quietly corrupted.
- Everything runs under `torch.no_grad()`.

The notebook checks all of this explicitly: BN layers report `training=False`, dropout layers report
`training=True` before sampling, and every BN running statistic is **bit-identical** before and after
the full run.

**The honest caveat: this is last-layer MC-dropout.** The architecture has exactly one dropout
layer, `fc[0]`, just before the final linear layer. Everything below it (the whole convolutional
backbone) is deterministic, so all 30 passes see **identical 512-d features** and differ only in
which features the head drops. §6 checks this directly. What this measures is therefore the
**classifier head's** uncertainty about how to read a fixed feature vector, not the uncertainty
of the full network. If the backbone extracts misleading features, e.g. for an out-of-distribution
scan, the head can be confidently wrong about them, and this method won't see it. Last-layer
dropout is a common, legitimate approximation (it's cheap, and it needs no retraining here), but
the write-up should say "head uncertainty", not "full epistemic uncertainty".

**Seeding.** The dropout masks are random across the 30 passes, which is the whole point. The *set*
of 30 passes is still reproducible: the RNG is seeded right before sampling and the loader order is
fixed, so a rerun draws the same masks in the same order (checked in §6).

In [ ]:
%%writefile /kaggle/working/src/uncertainty/mc_dropout.py
import torch
from torch import nn
from torch.nn.modules.batchnorm import _BatchNorm
from torch.nn.modules.dropout import _DropoutNd


def enable_mc_dropout(model):
    # eval everything (BN uses running stats), then re-enable only dropout. Never model.train().
    model.eval()
    n = 0
    for m in model.modules():
        if isinstance(m, _DropoutNd):
            m.train()
            n += 1
    return n


def mc_state(model):
    bn = [m for m in model.modules() if isinstance(m, _BatchNorm)]
    do = [m for m in model.modules() if isinstance(m, _DropoutNd)]
    return {"n_bn": len(bn), "bn_in_train": sum(m.training for m in bn),
            "n_dropout": len(do), "dropout_in_train": sum(m.training for m in do)}


def assert_mc_mode(model):
    s = mc_state(model)
    assert s["n_dropout"] > 0, "model has no dropout layer -- MC-dropout is meaningless"
    assert s["bn_in_train"] == 0, f"{s['bn_in_train']} BatchNorm layers in train mode"
    assert s["dropout_in_train"] == s["n_dropout"], "some dropout layers are not active"
    return s


def bn_snapshot(model):
    return {k: v.detach().clone() for k, v in model.state_dict().items()
            if k.endswith(("running_mean", "running_var", "num_batches_tracked"))}


def bn_unchanged(before, model):
    after = bn_snapshot(model)
    return before.keys() == after.keys() and all(torch.equal(before[k], after[k]) for k in before)


@torch.no_grad()
def mc_predict(model, loader, n_passes, device, seed):
    # returns softmax samples (T, N, C); one data load per batch, T passes over it
    assert_mc_mode(model)
    torch.manual_seed(seed)
    out = []
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        out.append(torch.stack([model(x).float().softmax(1) for _ in range(n_passes)]).cpu())
    return torch.cat(out, dim=1).numpy()


@torch.no_grad()
def deterministic_predict(model, loader, device):
    model.eval()
    return torch.cat([model(b["image"].to(device)).float().softmax(1).cpu() for b in loader]).numpy()

## 4. Uncertainty measures and evaluation metrics

For one image, let $p_t \in \Delta^{C-1}$ be the softmax output of pass $t = 1..T$, with $C = 3$
classes. Entropies are in nats; the maximum is $\ln 3 \approx 1.099$ (uniform over the 3 classes).

| quantity | definition | reads as |
|---|---|---|
| MC mean probability | $\bar p = \frac{1}{T}\sum_t p_t$ | the MC prediction is $\hat y = \arg\max_c \bar p_c$ |
| **predictive entropy** | $\mathcal{H}[\bar p] = -\sum_c \bar p_c \ln \bar p_c$ | **total** uncertainty: high if the averaged prediction is spread out, for whatever reason |
| **expected entropy** | $\frac{1}{T}\sum_t \mathcal{H}[p_t]$ | **aleatoric** part: each pass on its own is unsure, e.g. the image is genuinely ambiguous |
| **mutual information** (BALD) | $\mathcal{H}[\bar p] - \frac{1}{T}\sum_t \mathcal{H}[p_t] \;\ge 0$ | **epistemic** part: the passes are each confident but *disagree*, i.e. the model's (head's) knowledge is uncertain |
| predicted-class std | $\mathrm{std}_t\, p_{t,\hat y}$ | how much the winning class's probability wobbles between passes |
| deterministic confidence | $\max_c p^{\text{det}}_c$ with dropout **off** | the single-pass confidence notebook 1 reported |

MI ≥ 0 follows from Jensen's inequality. Tiny negative values from floating-point rounding are
clipped to 0.

**Calibration.**
- **ECE** (expected calibration error, Guo et al., 2017): 15 equal-width confidence bins; the
  average |accuracy − mean confidence| per bin, weighted by bin size. 0 means perfectly calibrated.
  With ~95% accuracy, most predictions land in the top bin, so ECE is noisy; that's why it gets a
  bootstrap CI like everything else.
- **NLL** (mean −ln p of the true class) and **Brier score** (mean squared error of the probability
  vector) are *proper scoring rules*. They don't depend on binning, so they back up ECE.

**Error detection.** For each uncertainty score, the AUROC for telling **wrong** predictions from
**right** ones. 0.5 means the score is useless for flagging errors, 1.0 means every error is more
uncertain than every correct prediction. MC scores are judged against the MC prediction's errors,
and the deterministic confidence against the deterministic prediction's errors.

**Patient-level bootstrap.** Exactly as in notebook 3: resample the ~35 test **patients** with
replacement, take all of each sampled patient's slices, and recompute the statistic. The 2.5–97.5
percentiles of 1000 resamples give the CI. Every summary number in this notebook gets one.

In [ ]:
%%writefile /kaggle/working/src/uncertainty/metrics.py
import numpy as np
from sklearn.metrics import roc_auc_score

EPS = 1e-12


def entropy(p, axis=-1):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1.0)
    return -(p * np.log(p)).sum(axis=axis)


def mc_decomposition(samples):
    # samples: (T, N, C) softmax outputs
    s = np.asarray(samples, dtype=np.float64)
    mean = s.mean(axis=0)
    pred = mean.argmax(axis=1)
    pe = entropy(mean)
    ee = entropy(s).mean(axis=0)
    return {
        "mean_probs": mean,
        "mc_pred": pred,
        "mc_mean_confidence": mean.max(axis=1),
        "predictive_entropy": pe,
        "expected_entropy": ee,
        "mutual_information": np.clip(pe - ee, 0.0, None),   # >= 0 by Jensen; clip float noise
        "predicted_class_std": s[:, np.arange(s.shape[1]), pred].std(axis=0),
    }


def reliability_bins(confidence, correct, n_bins=15):
    conf, corr = np.asarray(confidence, float), np.asarray(correct, float)
    idx = np.clip(np.ceil(conf * n_bins).astype(int) - 1, 0, n_bins - 1)   # bins (lo, hi]
    rows = []
    for b in range(n_bins):
        m = idx == b
        rows.append({"lo": b / n_bins, "hi": (b + 1) / n_bins, "count": int(m.sum()),
                     "accuracy": float(corr[m].mean()) if m.any() else float("nan"),
                     "confidence": float(conf[m].mean()) if m.any() else float("nan")})
    return rows


def ece(confidence, correct, n_bins=15):
    rows = reliability_bins(confidence, correct, n_bins)
    n = sum(r["count"] for r in rows)
    return float(sum(r["count"] * abs(r["accuracy"] - r["confidence"]) for r in rows if r["count"]) / n)


def nll(probs, y):
    p = np.asarray(probs, float)[np.arange(len(y)), np.asarray(y)]
    return float(-np.log(np.clip(p, EPS, 1.0)).mean())


def brier(probs, y):
    p = np.asarray(probs, float)
    onehot = np.eye(p.shape[1])[np.asarray(y)]
    return float(((p - onehot) ** 2).sum(axis=1).mean())


def auroc(score, is_error):
    e = np.asarray(is_error).astype(bool)
    if e.all() or not e.any():
        return float("nan")   # undefined without both classes (e.g. a bootstrap draw with no errors)
    return float(roc_auc_score(e, score))


def patient_bootstrap_indices(patient_ids, n_boot=1000, seed=42):
    # each draw: patients with replacement, all of a drawn patient's rows together
    pids = np.asarray(patient_ids)
    groups = [np.flatnonzero(pids == p) for p in np.unique(pids)]
    rng = np.random.default_rng(seed)
    return [np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))]) for _ in range(n_boot)]


def bootstrap_ci(fn, arrays, boot_indices, alpha=0.05):
    vals = np.array([fn(*(a[idx] for a in arrays)) for idx in boot_indices], dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return [float("nan"), float("nan")]
    return [float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2)))]

In [ ]:
%%writefile /kaggle/working/src/uncertainty/__init__.py
from src.uncertainty import mc_dropout, metrics

## 5. Sanity tests for the uncertainty code

Same approach as notebook 3: known-answer tests run before the real run, and the notebook stops if
one fails. They're saved as `tests/test_uncertainty_metrics.py`.

- **Entropy and decomposition:** uniform → ln 3, one-hot → 0. MI = 0 when all passes are identical;
  MI > 0 when confident passes disagree; MI ≤ predictive entropy.
- **ECE:** a synthetic *calibrated* model (correct with probability = its confidence) gets ECE ≈ 0;
  a model that's 20 points overconfident gets ECE ≈ 0.2.
- **MC mode, on a tiny conv + BatchNorm + Dropout net:** after `enable_mc_dropout`, BN is in eval,
  dropout in train, passes differ, and BN running stats are bit-identical after many passes. The
  same test shows the pitfall: calling `model.train()` instead *does* change the running stats.
- **Bootstrap:** a resampled patient always brings all of its rows.

In [ ]:
%%writefile /kaggle/working/tests/test_uncertainty_metrics.py
import numpy as np
import pytest
import torch
from torch import nn

from src.uncertainty import mc_dropout as MC
from src.uncertainty import metrics as U


def test_entropy_bounds():
    assert U.entropy(np.full(3, 1 / 3)) == pytest.approx(np.log(3))
    assert U.entropy(np.array([1.0, 0.0, 0.0])) == pytest.approx(0.0, abs=1e-9)


def test_decomposition_identical_passes_have_zero_mi():
    s = np.tile(np.array([[0.7, 0.2, 0.1]]), (30, 1, 1))     # (T, N=1, C)
    d = U.mc_decomposition(s)
    assert d["mutual_information"][0] == pytest.approx(0.0, abs=1e-9)
    assert d["predictive_entropy"][0] == pytest.approx(d["expected_entropy"][0])
    assert d["predicted_class_std"][0] == pytest.approx(0.0)


def test_disagreeing_confident_passes_are_epistemic():
    s = np.array([[[0.98, 0.01, 0.01]], [[0.01, 0.98, 0.01]]] * 15)   # each pass confident, they disagree
    d = U.mc_decomposition(s)
    assert d["mutual_information"][0] > 0.5
    assert d["expected_entropy"][0] < 0.15
    assert d["mutual_information"][0] <= d["predictive_entropy"][0] + 1e-12


def test_mi_nonnegative_and_bounded_random():
    rng = np.random.default_rng(0)
    s = rng.dirichlet(np.ones(3), size=(30, 200))
    d = U.mc_decomposition(s)
    assert (d["mutual_information"] >= 0).all()
    assert (d["mutual_information"] <= d["predictive_entropy"] + 1e-12).all()


def test_ece_calibrated_vs_overconfident():
    rng = np.random.default_rng(0)
    conf = rng.uniform(0.4, 1.0, 200_000)
    calibrated = rng.random(conf.size) < conf
    overconf = rng.random(conf.size) < np.clip(conf - 0.2, 0, 1)
    assert U.ece(conf, calibrated) < 0.01
    assert U.ece(conf, overconf) == pytest.approx(0.2, abs=0.02)


def test_proper_scores():
    y = np.array([0, 1, 2])
    perfect = np.eye(3)
    assert U.nll(perfect, y) == pytest.approx(0.0, abs=1e-9)
    assert U.brier(perfect, y) == pytest.approx(0.0)
    assert U.brier(np.full((3, 3), 1 / 3), y) == pytest.approx(2 / 3)


def test_auroc_edges():
    assert U.auroc([0.1, 0.2, 0.9, 0.8], [0, 0, 1, 1]) == 1.0
    assert np.isnan(U.auroc([0.1, 0.2], [0, 0]))


def test_bootstrap_keeps_patients_whole():
    pids = np.array(["a", "a", "a", "b", "c", "c"])
    for idx in U.patient_bootstrap_indices(pids, n_boot=50, seed=1):
        for p in np.unique(pids[idx]):
            assert (pids[idx] == p).sum() % (pids == p).sum() == 0


def _tiny_net():
    torch.manual_seed(0)
    net = nn.Sequential(nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                        nn.Flatten(), nn.Dropout(0.5), nn.Linear(8, 3))
    net.train()
    with torch.no_grad():                      # give BN non-trivial running stats
        for _ in range(5):
            net(torch.randn(16, 3, 12, 12))
    return net


def test_enable_mc_dropout_keeps_bn_frozen():
    net = _tiny_net()
    n = MC.enable_mc_dropout(net)
    s = MC.assert_mc_mode(net)
    assert n == 1 and s["n_bn"] == 1 and not net.training
    before = MC.bn_snapshot(net)
    x = torch.randn(4, 3, 12, 12)
    with torch.no_grad():
        outs = torch.stack([net(x) for _ in range(20)])
    assert outs.std(0).mean() > 0                  # passes actually differ
    assert MC.bn_unchanged(before, net)


def test_model_train_would_corrupt_bn():
    net = _tiny_net()
    before = MC.bn_snapshot(net)
    net.train()                                     # the mistake MC-dropout must avoid
    with torch.no_grad():
        net(torch.randn(4, 3, 12, 12))
    assert not MC.bn_unchanged(before, net)
    with pytest.raises(AssertionError):
        MC.assert_mc_mode(net)


def test_mc_predict_seeded():
    net = _tiny_net()
    MC.enable_mc_dropout(net)
    loader = [{"image": torch.randn(5, 3, 12, 12)}, {"image": torch.randn(3, 3, 12, 12)}]
    a = MC.mc_predict(net, loader, 10, "cpu", seed=7)
    b = MC.mc_predict(net, loader, 10, "cpu", seed=7)
    c = MC.mc_predict(net, loader, 10, "cpu", seed=8)
    assert a.shape == (10, 8, 3) and np.allclose(a.sum(-1), 1)
    assert np.array_equal(a, b) and not np.array_equal(a, c)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_uncertainty_metrics.py"],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "uncertainty sanity tests failed -- fix src/uncertainty before trusting any number below"

## 6. Model, test set, and checks before sampling

`load_checkpoint` from notebook 1 builds the model with `pretrained=False` (no ImageNet download)
and returns it in eval mode. If notebook 1's `classifier.py` ever gains its own
`enable_mc_dropout`, it's used; otherwise the one in `src/uncertainty/` is. The test dataset is
built the same way notebook 1 built it, and its order is checked against `test_predictions.csv`
row by row.

`tumor_pixel_count` and the original image size come from the manifest. Their ratio gives
`tumor_frac`, the fraction of the image covered by tumor, which is comparable across the 512 and
256 px images.

In [ ]:
from src.utils.seed import set_seed
from src.models import classifier as C
from src.data.dataset import BrainTumorDataset
from src.data.transforms import get_eval_transforms
from src.uncertainty import mc_dropout as MC, metrics as U

model, ckpt = C.load_checkpoint(CKPT_PATH, DEVICE)
CFG = ckpt["config"]
CLASS_NAMES = ckpt["class_names"]
SEED = CFG["seed"]
T = CFG.get("uncertainty", {}).get("n_forward_passes", 30)
N_BOOT = CFG.get("uncertainty", {}).get("n_bootstrap", 1000)
set_seed(SEED)

enable_mc_dropout = getattr(C, "enable_mc_dropout", MC.enable_mc_dropout)
print("enable_mc_dropout from:", enable_mc_dropout.__module__)
print("head:", model.fc, "| T =", T)

manifest = pd.read_csv(MANIFEST_PATH)
key = pd.read_csv(NB1_DIR / "results" / "test_predictions.csv")
with open(NB1_DIR / "results" / "test_metrics.json") as f:
    nb1_metrics = json.load(f)
test_ds = BrainTumorDataset(manifest, DATA_ROOT, "test", get_eval_transforms(CFG["image_size"]))
assert list(test_ds.df.file_path) == list(key.file_path), "test order differs from test_predictions.csv"
N = len(key)
loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

size = test_ds.df[["file_path", "tumor_pixel_count", "image_h", "image_w"]].copy()
tumor_frac = (size.tumor_pixel_count / (size.image_h * size.image_w)).to_numpy()
true = key.true.to_numpy()
pids = key.patient_id.to_numpy()
print(f"{N} test images, {len(np.unique(pids))} patients")

### Deterministic pass (dropout off)

This is the baseline the MC numbers are compared against, and a check that the right weights
loaded. Agreement with notebook 1's predictions should be ≥ 99%. It may not be exactly 100%
because notebook 1 predicted under fp16 autocast and this runs in fp32, so near-ties can flip.

In [ ]:
det_probs = MC.deterministic_predict(model, loader, DEVICE)
det_pred = det_probs.argmax(1)
det_conf = det_probs.max(1)
agree = (det_pred == key.pred.to_numpy()).mean()
print(f"agreement with notebook 1: {agree:.4f} | det accuracy {(det_pred == true).mean():.4f} "
      f"(notebook 1 reported {nb1_metrics['accuracy']:.4f})")
assert agree >= 0.99, "checkpoint does not reproduce notebook 1's predictions"

### Switch on MC mode, and check it before using it

1. `enable_mc_dropout`, then assert: BN in eval, dropout in train, top-level model not in training.
2. Report **where** the dropout layers are. The last-layer caveat in §3 assumes exactly one, at
   `fc.0`; if notebook 1's architecture changes, this cell shows it.
3. Snapshot every BN running statistic, to compare after the full run.
4. **The passes really differ:** two passes on one batch must give different outputs. If they
   didn't, dropout wouldn't be active, and every "uncertainty" below would be 0.
5. **Features are identical across passes:** a hook on `avgpool` captures the 512-d input to the
   head on two passes. Identical features are the direct evidence that this is head-only
   uncertainty.
6. **The set of passes is seeded:** the same seed gives bit-identical samples, a different seed
   gives different ones.

In [ ]:
n_do = enable_mc_dropout(model)
state = MC.assert_mc_mode(model)
assert not model.training
dropout_where = [name for name, m in model.named_modules() if isinstance(m, torch.nn.modules.dropout._DropoutNd)]
print("MC state:", state, "| dropout layers:", dropout_where)
if dropout_where != ["fc.0"]:
    print("NOTE: dropout is not exactly one layer at fc.0 -- revisit the last-layer caveat in section 3")

BN_BEFORE = MC.bn_snapshot(model)

xb = next(iter(loader))["image"][:16].to(DEVICE)
feats = []
hook = model.avgpool.register_forward_hook(lambda m, i, o: feats.append(o.detach().flatten(1)))
with torch.no_grad():
    o1, o2 = model(xb).softmax(1), model(xb).softmax(1)
hook.remove()
print(f"pass-to-pass max |delta p| = {(o1 - o2).abs().max():.4f} (must be > 0)")
print(f"pass-to-pass max |delta feature| = {(feats[0] - feats[1]).abs().max():.2e} (0 => head-only uncertainty)")
assert (o1 - o2).abs().max() > 0, "dropout not active -- passes are identical"

small = [{"image": xb}]
s_a, s_b, s_c = (MC.mc_predict(model, small, 5, DEVICE, seed) for seed in [SEED, SEED, SEED + 1])
print("same seed identical:", np.array_equal(s_a, s_b), "| different seed differs:", not np.array_equal(s_a, s_c))
assert np.array_equal(s_a, s_b) and not np.array_equal(s_a, s_c)

## 7. Sampling

`T` passes per batch: each batch is loaded once and pushed through the model `T` times, so the
`.mat` decoding isn't repeated 30 times. The RNG is seeded right before the run. After it, the BN
running statistics are compared bit for bit with the snapshot from §6.

In [ ]:
t0 = time.time()
samples = MC.mc_predict(model, loader, T, DEVICE, seed=SEED)      # (T, N, C)
elapsed = time.time() - t0
print(f"{T} passes x {N} images in {elapsed:.1f}s; samples {samples.shape}")

assert samples.shape == (T, N, len(CLASS_NAMES)) and np.allclose(samples.sum(-1), 1, atol=1e-4)
assert MC.bn_unchanged(BN_BEFORE, model), "BatchNorm running statistics changed during MC sampling"
MC.assert_mc_mode(model)
print("BatchNorm running stats unchanged after sampling; BN still in eval, dropout still active")
model.eval()   # back to plain inference so nothing downstream inherits MC mode by accident

## 8. Per-image results → `uncertainty_results.csv`

One row per test image, in `test_predictions.csv` order. The requested columns come first; after
them, the per-class MC and deterministic probabilities, `tumor_frac` and class names, which
notebook 5 will want. The raw `(T, N, C)` samples go to `mc_samples.npz`, so any other measure can
be computed later without re-sampling.

In [ ]:
d = U.mc_decomposition(samples)
mc_pred = d["mc_pred"]
res = pd.DataFrame({
    "file_path": key.file_path, "patient_id": key.patient_id, "true": true,
    "mc_pred": mc_pred, "det_pred": det_pred,
    "det_confidence": det_conf, "mc_mean_confidence": d["mc_mean_confidence"],
    "predictive_entropy": d["predictive_entropy"], "expected_entropy": d["expected_entropy"],
    "mutual_information": d["mutual_information"], "predicted_class_std": d["predicted_class_std"],
    "correct_mc": mc_pred == true, "correct_det": det_pred == true,
})
for k, c in enumerate(CLASS_NAMES):
    res[f"p_mc_{c}"] = d["mean_probs"][:, k]
for k, c in enumerate(CLASS_NAMES):
    res[f"p_det_{c}"] = det_probs[:, k]
res["true_name"] = [CLASS_NAMES[i] for i in true]
res["tumor_frac"] = tumor_frac

res.to_csv(UNC_DIR / "uncertainty_results.csv", index=False)
np.savez_compressed(UNC_DIR / "mc_samples.npz", samples=samples.astype(np.float32), det_probs=det_probs.astype(np.float32),
                    file_path=key.file_path.to_numpy().astype(str))
res.describe().T[["mean", "std", "min", "50%", "max"]].round(4)

## 9. Analyses

Every number below gets a patient-bootstrap 95% CI (`N_BOOT` resamples). The same resample indices
are reused for every statistic, so all of them are computed on identical bootstrap draws and the
CIs are directly comparable. `report()` stores each value and its CI in the summary.

In [ ]:
BOOT = U.patient_bootstrap_indices(pids, n_boot=N_BOOT, seed=SEED)
SUMMARY = {}


def report(section, name, fn, *arrays):
    arrays = tuple(np.asarray(a) for a in arrays)
    value = float(fn(*arrays))
    ci = U.bootstrap_ci(fn, arrays, BOOT)
    SUMMARY.setdefault(section, {})[name] = {"value": value, "ci95_patient_bootstrap": ci}
    return f"{value:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]"


def acc(pred, y):
    return (pred == y).mean()

### 9a. Deterministic vs MC-averaged

With dropout only in the head, the MC average and the deterministic pass should mostly agree:
averaging over dropout masks approximates the deterministic "weight-scaled" network. The questions
are whether MC **changes accuracy** (it rarely moves by more than a point), and whether it changes
**which images look confident**. If the least-confident 10% are mostly the same images under both,
MC adds little beyond a single pass for flagging cases. If the sets differ a lot, the choice matters.

In [ ]:
print("accuracy  det:", report("accuracy", "deterministic", acc, det_pred, true))
print("accuracy   MC:", report("accuracy", "mc", acc, mc_pred, true))
print("accuracy MC - det:", report("accuracy", "mc_minus_det", lambda a, b, y: acc(a, y) - acc(b, y), mc_pred, det_pred, true))
print(f"predictions that differ between det and MC: {(mc_pred != det_pred).sum()} of {N}")

rho = spearmanr(det_conf, d["mc_mean_confidence"])[0]
k10 = max(1, int(round(0.10 * N)))
low_det, low_mc = set(np.argsort(det_conf)[:k10]), set(np.argsort(d["mc_mean_confidence"])[:k10])
jacc = len(low_det & low_mc) / len(low_det | low_mc)
print(f"Spearman(det conf, MC conf) = {rho:.3f}; overlap of the 10% least-confident sets (Jaccard) = {jacc:.2f}")
SUMMARY["det_vs_mc"] = {"n_pred_differ": int((mc_pred != det_pred).sum()), "spearman_confidence": float(rho),
                        "jaccard_least_confident_10pct": float(jacc)}

fig, ax = plt.subplots(figsize=(5.2, 5))
diff = mc_pred != det_pred
ax.scatter(det_conf[~diff], d["mc_mean_confidence"][~diff], s=10, alpha=0.5, color="#555", label="same prediction")
ax.scatter(det_conf[diff], d["mc_mean_confidence"][diff], s=40, marker="x", color="crimson", label="prediction changes")
ax.plot([1 / 3, 1], [1 / 3, 1], "k--", lw=0.8)
ax.set_xlabel("deterministic confidence (dropout off)"); ax.set_ylabel("MC mean confidence")
ax.set_title(f"Spearman {rho:.3f}"); ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / "uncertainty_det_vs_mc_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

### 9b. Calibration

The reliability diagram plots each bin's accuracy against its mean confidence. Bars below the
diagonal mean **over**confidence: the model says 95%, but it's right less often than that. The
grey histogram underneath shows how many predictions fall in each bin. Expect nearly all of them in
the top one or two bins, which is why the CIs on ECE are wide.

MC averaging usually *lowers* confidence a little, which tends to help an overconfident model.
Whether it helps *this* model is what the numbers decide. The "better calibrated" verdict printed
below only claims a winner when the CI of the difference excludes 0.

In [ ]:
mc_probs = d["mean_probs"]
for name, probs, pred_ in [("deterministic", det_probs, det_pred), ("mc", mc_probs, mc_pred)]:
    conf_ = probs.max(1)
    print(f"{name:13s} ECE {report('calibration', f'{name}_ece', U.ece, conf_, pred_ == true)}"
          f" | NLL {report('calibration', f'{name}_nll', U.nll, probs, true)}"
          f" | Brier {report('calibration', f'{name}_brier', U.brier, probs, true)}")

dece = report("calibration", "ece_mc_minus_det", lambda pm, pd_, y: U.ece(pm.max(1), pm.argmax(1) == y) - U.ece(pd_.max(1), pd_.argmax(1) == y),
              mc_probs, det_probs, true)
dnll = report("calibration", "nll_mc_minus_det", lambda pm, pd_, y: U.nll(pm, y) - U.nll(pd_, y), mc_probs, det_probs, true)
print("ECE MC - det:", dece, "| NLL MC - det:", dnll)
ci = SUMMARY["calibration"]["ece_mc_minus_det"]["ci95_patient_bootstrap"]
verdict = ("MC-averaged is better calibrated (ECE CI below 0)" if ci[1] < 0 else
           "deterministic is better calibrated (ECE CI above 0)" if ci[0] > 0 else
           "no calibration difference distinguishable from patient sampling noise (ECE CI spans 0)")
SUMMARY["calibration"]["verdict"] = verdict
print("verdict:", verdict)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
for col, (name, probs, pred_, color) in enumerate([("deterministic (dropout off)", det_probs, det_pred, "#7570b3"),
                                                    (f"MC-averaged (T={T})", mc_probs, mc_pred, "#e7298a")]):
    bins = U.reliability_bins(probs.max(1), pred_ == true)
    centers = [(b["lo"] + b["hi"]) / 2 for b in bins]
    width = bins[0]["hi"] - bins[0]["lo"]
    axes[0, col].bar(centers, [b["accuracy"] if b["count"] else 0 for b in bins], width=width * 0.95, color=color, alpha=0.75, label="accuracy")
    axes[0, col].plot([0, 1], [0, 1], "k--", lw=0.8)
    for b, cx in zip(bins, centers):
        if b["count"]:
            axes[0, col].plot([cx, cx], [b["accuracy"], b["confidence"]], color="crimson", lw=1.5)
    key_ece = "deterministic_ece" if col == 0 else "mc_ece"
    e = SUMMARY["calibration"][key_ece]
    axes[0, col].set_title(f"{name}\nECE {e['value']:.3f} [{e['ci95_patient_bootstrap'][0]:.3f}, {e['ci95_patient_bootstrap'][1]:.3f}]", fontsize=10)
    axes[0, col].set_ylabel("accuracy in bin"); axes[0, col].set_ylim(0, 1.02)
    axes[1, col].bar(centers, [b["count"] for b in bins], width=width * 0.95, color="#999")
    axes[1, col].set_xlabel("confidence"); axes[1, col].set_ylabel("count"); axes[1, col].set_yscale("log")
axes[0, 0].set_xlim(1 / 3 - 0.02, 1.0)
fig.suptitle("Reliability diagrams (red = gap to perfect calibration)", y=1.0)
plt.tight_layout()
fig.savefig(FIG_DIR / "uncertainty_reliability.png", dpi=150, bbox_inches="tight")
plt.show()

### 9c. Uncertainty as an error detector

For each score, the AUROC for telling the model's **own** errors from its correct predictions. The
deterministic baseline is `1 − det_confidence`, i.e. "just use the softmax". For MC to earn its 30×
cost, its scores should beat that baseline, and the CI of the difference has to exclude 0 before
calling it a real improvement.

With only ~20 test errors, these AUROCs are noisy. A bootstrap draw with no errors at all gives an
undefined AUROC and is skipped. The number of usable draws is reported.

The **confident-and-wrong** table at the end lists errors the model made with MC confidence ≥ 0.9.
These are the cases that matter clinically: no uncertainty threshold would have sent them for review.

In [ ]:
wrong_mc, wrong_det = ~res.correct_mc.to_numpy(), ~res.correct_det.to_numpy()
SCORES = {
    "predictive_entropy": (d["predictive_entropy"], wrong_mc),
    "mutual_information": (d["mutual_information"], wrong_mc),
    "expected_entropy": (d["expected_entropy"], wrong_mc),
    "predicted_class_std": (d["predicted_class_std"], wrong_mc),
    "1-mc_mean_confidence": (1 - d["mc_mean_confidence"], wrong_mc),
    "1-det_confidence (baseline)": (1 - det_conf, wrong_det),
}
print(f"errors: MC {wrong_mc.sum()}, det {wrong_det.sum()} of {N}")
for name, (score, wrong) in SCORES.items():
    print(f"AUROC {name:30s} {report('error_detection_auroc', name, U.auroc, score, wrong)}")
usable = sum(1 for idx in BOOT if 0 < wrong_mc[idx].sum() < len(idx))
SUMMARY["error_detection_auroc"]["n_usable_bootstrap_draws"] = usable
print(f"({usable} of {N_BOOT} bootstrap draws contained errors)")
print("AUROC entropy - baseline:", report("error_detection_auroc", "predictive_entropy_minus_det_baseline",
      lambda s1, w1, s2, w2: U.auroc(s1, w1) - U.auroc(s2, w2), d["predictive_entropy"], wrong_mc, 1 - det_conf, wrong_det))

for name in ["predictive_entropy", "mutual_information", "expected_entropy", "predicted_class_std"]:
    s = res[name].to_numpy()
    c_ = report("by_correctness", f"{name}/correct", lambda v, w: v[~w].mean(), s, wrong_mc)
    w_ = report("by_correctness", f"{name}/incorrect", lambda v, w: v[w].mean() if w.any() else np.nan, s, wrong_mc)
    print(f"mean {name:20s} correct {c_} | wrong {w_}")

danger = res[(~res.correct_mc) & (res.mc_mean_confidence >= 0.9)]
SUMMARY["confident_and_wrong"] = {"threshold": 0.9, "count": int(len(danger)), "of_errors": int(wrong_mc.sum()),
                                  "file_paths": danger.file_path.tolist()}
print(f"\nconfident-and-wrong (MC conf >= 0.9): {len(danger)} of {wrong_mc.sum()} errors")
display(danger[["file_path", "patient_id", "true", "mc_pred", "mc_mean_confidence", "predictive_entropy", "mutual_information"]]
        .assign(true=lambda t: [CLASS_NAMES[i] for i in t.true], mc_pred=lambda t: [CLASS_NAMES[i] for i in t.mc_pred])
        .sort_values("mc_mean_confidence", ascending=False))

In [ ]:
measures = ["predictive_entropy", "mutual_information", "expected_entropy", "predicted_class_std"]
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for ax, m in zip(axes[0], measures):
    v = res[m].to_numpy()
    data = [v[~wrong_mc], v[wrong_mc]]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False)
    for patch, col in zip(bp["boxes"], ["#888", "crimson"]):
        patch.set_facecolor(col); patch.set_alpha(0.45)
    for k, (vals, col) in enumerate(zip(data, ["#555", "crimson"]), start=1):
        ax.scatter(k + np.random.default_rng(k).uniform(-0.15, 0.15, len(vals)), vals, s=5 if k == 1 else 18, color=col, alpha=0.45)
    a = SUMMARY["error_detection_auroc"][m]
    ax.set_xticks([1, 2], [f"correct (n={len(data[0])})", f"wrong (n={len(data[1])})"])
    ax.set_title(f"{m}\nAUROC {a['value']:.3f} [{a['ci95_patient_bootstrap'][0]:.2f}, {a['ci95_patient_bootstrap'][1]:.2f}]", fontsize=9)
    if m in ("predictive_entropy", "mutual_information", "expected_entropy"):
        ax.set_yscale("symlog", linthresh=1e-3)

ax = axes[1, 0]
for name, (score, wrong) in SCORES.items():
    if wrong.any() and not wrong.all():
        fpr, tpr, _ = roc_curve(wrong, score)
        ax.plot(fpr, tpr, lw=1.5, ls="--" if "baseline" in name else "-", label=f"{name} ({SUMMARY['error_detection_auroc'][name]['value']:.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xlabel("false positive rate (correct flagged)"); ax.set_ylabel("true positive rate (errors caught)")
ax.set_title("ROC: flagging errors by uncertainty", fontsize=9); ax.legend(fontsize=7, frameon=False)

ax = axes[1, 1]
ax.scatter(res.mc_mean_confidence[~wrong_mc], res.mutual_information[~wrong_mc], s=8, alpha=0.4, color="#555", label="correct")
ax.scatter(res.mc_mean_confidence[wrong_mc], res.mutual_information[wrong_mc], s=30, marker="x", color="crimson", label="wrong")
ax.axvline(0.9, color="k", lw=0.8, ls="--")
ax.set_xlabel("MC mean confidence"); ax.set_ylabel("mutual information"); ax.set_yscale("symlog", linthresh=1e-3)
ax.set_title("confident-and-wrong = red x right of the line", fontsize=9); ax.legend(fontsize=7, frameon=False)
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()
fig.savefig(FIG_DIR / "uncertainty_error_detection.png", dpi=150, bbox_inches="tight")
plt.show()

### 9d. By class and by tumor size

If one class is systematically less certain, the uncertainty partly encodes class identity, not
per-image difficulty. That would matter for notebook 5, where a "low-confidence" group could end up
being mostly one class. Tumor size gets the same check. Size bins are **tertiles** of `tumor_frac`
over the whole test set, so each bin has about a third of the images.

Class and size are confounded here: pituitary tumors are small. The per-class table shows both, and
the scatter colours points by class, so a size effect that's really a class effect is visible.

In [ ]:
edges = np.quantile(tumor_frac, [0, 1 / 3, 2 / 3, 1])
size_bin = np.clip(np.searchsorted(edges, tumor_frac, side="right") - 1, 0, 2)
SIZE_NAMES = ["small", "medium", "large"]
res["size_tertile"] = [SIZE_NAMES[b] for b in size_bin]
SUMMARY["size_tertile_edges_tumor_frac"] = edges.tolist()

rows = []
for group_name, labels, names in [("by_class", true, CLASS_NAMES), ("by_size", size_bin, SIZE_NAMES)]:
    for k, gname in enumerate(names):
        row = {"group": f"{group_name}:{gname}", "n": int((labels == k).sum())}
        for m in ["predictive_entropy", "mutual_information", "mc_mean_confidence"]:
            row[m] = report(group_name, f"{gname}/{m}", lambda v, g, kk=k: v[g == kk].mean() if (g == kk).any() else np.nan,
                            res[m].to_numpy(), labels)
        row["mc_accuracy"] = report(group_name, f"{gname}/mc_accuracy", lambda p, y, g, kk=k: (p[g == kk] == y[g == kk]).mean() if (g == kk).any() else np.nan,
                                    mc_pred, true, labels)
        row["median_tumor_frac"] = float(np.median(tumor_frac[labels == k]))
        rows.append(row)
display(pd.DataFrame(rows).set_index("group"))
print("Spearman(tumor_frac, predictive entropy):",
      report("by_size", "spearman_tumor_frac_vs_entropy", lambda f, e: spearmanr(f, e)[0], tumor_frac, d["predictive_entropy"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
data = [res.loc[res.true == k, "predictive_entropy"].to_numpy() for k in range(len(CLASS_NAMES))]
bp = axes[0].boxplot(data, patch_artist=True, widths=0.55, showfliers=False)
cls_colors = ["#1b9e77", "#d95f02", "#7570b3"]
for patch, col in zip(bp["boxes"], cls_colors):
    patch.set_facecolor(col); patch.set_alpha(0.45)
for k, (vals, col) in enumerate(zip(data, cls_colors), start=1):
    axes[0].scatter(k + np.random.default_rng(k).uniform(-0.15, 0.15, len(vals)), vals, s=6, color=col, alpha=0.5)
axes[0].set_xticks(range(1, len(CLASS_NAMES) + 1), CLASS_NAMES); axes[0].set_yscale("symlog", linthresh=1e-3)
axes[0].set_ylabel("predictive entropy (nats)"); axes[0].set_title("uncertainty by true class", fontsize=10)
for k, (cname, col) in enumerate(zip(CLASS_NAMES, cls_colors)):
    m_ = true == k
    axes[1].scatter(tumor_frac[m_], d["predictive_entropy"][m_], s=10, alpha=0.55, color=col, label=cname)
for e in edges[1:-1]:
    axes[1].axvline(e, color="k", lw=0.7, ls=":")
axes[1].set_xscale("log"); axes[1].set_yscale("symlog", linthresh=1e-3)
axes[1].set_xlabel("tumor area / image area (log); dotted = tertile edges"); axes[1].set_ylabel("predictive entropy")
axes[1].set_title("uncertainty vs tumor size", fontsize=10); axes[1].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "uncertainty_by_class_and_size.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Summary, lineage, and reload check

`uncertainty_summary.json` records the lineage notebook 5 needs before joining tables: the
checkpoint MD5 (which notebook 2's `meta.json` also records for the explanations) and the manifest
MD5. Notebook 5 should refuse to join the uncertainty, audit and prediction tables unless all of
them agree.

In [ ]:
overall = {}
for m in ["predictive_entropy", "expected_entropy", "mutual_information", "predicted_class_std", "mc_mean_confidence", "det_confidence"]:
    overall[m] = report("overall_means", m, np.mean, res[m].to_numpy())

summary = {
    "lineage": {"checkpoint_md5": CKPT_MD5, "manifest_md5": WANT_MD5, "checkpoint": str(CKPT_PATH),
                "row_key": "test_predictions.csv row order, keyed by file_path"},
    "settings": {"n_forward_passes": T, "seed": SEED, "dropout_layers": dropout_where, "dropout_p": CFG["model"]["dropout_p"],
                 "mc_type": "last-layer (head-only) MC-dropout", "ece_bins": 15, "n_bootstrap": N_BOOT,
                 "bootstrap_unit": "patient", "entropy_units": "nats"},
    "n_test": int(N), "n_patients": int(len(np.unique(pids))),
    "agreement_det_with_nb1": float(agree),
    **SUMMARY,
}
with open(UNC_DIR / "uncertainty_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

back = pd.read_csv(UNC_DIR / "uncertainty_results.csv")
required = ["file_path", "patient_id", "true", "mc_pred", "det_pred", "det_confidence", "mc_mean_confidence",
            "predictive_entropy", "expected_entropy", "mutual_information", "predicted_class_std", "correct_mc", "correct_det"]
assert list(back.columns[:len(required)]) == required, back.columns[:len(required)].tolist()
assert len(back) == len(key) and (back.file_path == key.file_path).all(), "row count/order differs from test_predictions.csv"
assert (back.true == key.true).all()
assert np.allclose(back.predictive_entropy, d["predictive_entropy"]) and back[required].notna().all().all()
z = np.load(UNC_DIR / "mc_samples.npz")
assert z["samples"].shape == (T, N, len(CLASS_NAMES)) and (z["file_path"] == key.file_path.to_numpy()).all()
print(f"uncertainty_results.csv: {len(back)} rows, order matches test_predictions.csv; summary + samples saved")
print(json.dumps({k: summary[k] for k in ["lineage", "settings", "accuracy"]}, indent=2))
!ls -la /kaggle/working/results/uncertainty

## Interpreting the results

| axis | trustworthy | the danger case |
|---|---|---|
| **calibration** | ECE small (a few %), reliability bars close to the diagonal, NLL/Brier agree | bars well below the diagonal in the high-confidence bins: says 99%, is right 90% of the time |
| **error detection** | AUROC well above 0.5 (≥ 0.8 is useful), errors clearly shifted towards high entropy/MI | AUROC near 0.5: uncertainty doesn't separate right from wrong |
| **confident-and-wrong** | few or none of the errors have confidence ≥ 0.9 | many errors at ≥ 0.9: the model is *confidently wrong*, and no threshold catches it |
| **structure** | uncertainty roughly even across classes and sizes, apart from genuinely harder cases | one class or the smallest tumors carry most of the uncertainty *and* most of the errors |

How to read the components:
- **High expected entropy, low MI:** each pass is unsure in the same way. The image itself is
  ambiguous (aleatoric), and more data of the same kind wouldn't fix it.
- **High MI:** the passes are individually sure but disagree. The head's knowledge is uncertain
  (epistemic). These are the images where MC adds information a single pass can't give.
- If MI is tiny everywhere and almost all uncertainty is expected entropy, last-layer dropout
  isn't adding much beyond the softmax. That's a legitimate finding, and it fits the head-only
  caveat below.

**The last-layer caveat, again.** All 30 passes share identical backbone features (checked in §6).
These numbers describe how unsure the *classifier head* is about a fixed feature vector. They can't
reveal a backbone that extracts confidently wrong features, e.g. on scans unlike the training
distribution. Say "head-level MC-dropout uncertainty" in the write-up, and name full-network
dropout, deep ensembles or test-time augmentation as the stronger alternatives.

**Small-sample caveat.** ~35 test patients and ~20 errors mean wide CIs, especially on ECE and the
AUROCs. A difference whose patient-bootstrap CI includes 0 shouldn't be reported as a finding.

## Closing checklist

- [ ] Sanity tests passed (§5).
- [ ] Deterministic predictions agree with notebook 1 (≥ 99%).
- [ ] MC mode: BN in eval, dropout active, passes differ, **features identical across passes**,
  and **BN running stats unchanged** after sampling (all asserted).
- [ ] Det and MC accuracy within about a point of each other.
- [ ] `uncertainty_results.csv` reloads with the required columns first, in `test_predictions.csv`
  order (asserted).
- [ ] Look at the confident-and-wrong table. Those file paths are prime candidates for notebook 5's
  failure-case figures: do their explanations (from notebook 3) look faithful?

**Save & Run All**
1. **Save Version → Save & Run All (Commit) → Save**, with the GPU on and notebooks 0 and 1 still
   attached.
2. Check the version's **Output** tab for `results/uncertainty/` (`uncertainty_results.csv`,
   `uncertainty_summary.json`, `mc_samples.npz`), the `uncertainty_*.png` figures,
   `src/uncertainty/` and `tests/`.
3. **Notebook 5 (joint analysis)** attaches notebook 3's output (audit, which already carries
   notebook 2's `src/explain`) and this notebook's output (uncertainty). Join on `file_path` only
   after checking that the `checkpoint_md5` and `manifest_md5` in this summary match notebook 3's
   `audit_config.json`. Note that each of the two outputs has its own `src/`, so pick **one** for
   `sys.path`, or merge them deliberately in notebook 5.